In [1]:
import pandas as pd
import math
import matplotlib.pyplot as plt

# Charger les données
data = pd.read_csv("../data/diabetes.csv")

# on enlève la colonne target pour travailler seulement avec les features
X = data.drop(columns=[data.columns[-1]])
# conversion en liste 
X = X.values.tolist()
        
# paramètres de DBSCAN
eps = 40 # distance pour dire qu'un point est voisin
minPts = 3 #densité minimale

# fonction de distance
# ici on calcule la distance entre deux points
# on utilise la formule euclidienne classique
def distance(a, b):
    s = 0
    for i in range(len(a)):
        s += (a[i] - b[i]) ** 2
    return math.sqrt(s)

# trouver les voisins
# cette fonction cherche tous les points proches d’un point donné
def find_neighbors(X, idx, eps):
    neighbors = []
    # on parcourt tous les points du dataset
    for i in range(len(X)):
        # on calcule la distance entre le point de référence (idx)
        # et le point courant i
        # si cette distance est inférieure ou égale à eps,
        # alors les deux points sont considérés comme voisins
        if distance(X[idx], X[i]) <= eps:
            neighbors.append(i)
    # à la fin on retourne la liste des indices des points voisins
    return neighbors
    
#  expansion du cluster
# cette fonction sert à agrandir un cluster à partir d’un point core
def expand_cluster(X, labels, idx, neighbors, cluster_id):
     # on commence par dire que le point de départ appartient au cluster actuel
    labels[idx] = cluster_id
    # on utilise un index pour parcourir la liste des voisins
    i = 0
    # tant qu’on n’a pas exploré tous les voisins
    while i < len(neighbors):
        # on prend un point voisin
        p = neighbors[i]
        # si le point était considéré comme bruit (-1)
        # on le récupère et on l’ajoute au cluster
        if labels[p] == -1:
            labels[p] = cluster_id

        # si le point n’a jamais été visité (0)
        elif labels[p] == 0:
            # on l’ajoute au cluster
            labels[p] = cluster_id

            # on cherche ses propres voisins
            new_neighbors = find_neighbors(X, p, eps)
            # si ce point est dense (assez de voisins)
            # alors il peut étendre le cluster
            if len(new_neighbors) >= minPts:
                neighbors += new_neighbors

        # on passe au voisin suivant
        i += 1

        
#  DBSCAN principal
# ici on construit tous les clusters
def dbscan(X, eps, minPts):
    # 0 = non visité, -1 = bruit
    labels = [0] * len(X)
    cluster_id = 0
    
    for i in range(len(X)):
        # si le point est déjà visité, on passe au suivant
        if labels[i] != 0:
            continue
        # on récupère les voisins
        neighbors = find_neighbors(X, i, eps)
        # si pas assez de voisins, c’est du bruit
        
        if len(neighbors) < minPts:
            labels[i] = -1
        else:
            # sinon on crée un nouveau cluster
            cluster_id += 1
            expand_cluster(X, labels, i, neighbors, cluster_id)

    return labels

#  exécution DBSCAN
labels = dbscan(X, eps, minPts)
# on ajoute les clusters dans le dataframe
data["cluster"] = labels

#  résultat
print(data["cluster"].value_counts())
print("DBSCAN terminé")

cluster
 1    743
-1     10
 3     10
 2      5
Name: count, dtype: int64
DBSCAN terminé
